# exp12: J++, J-lens and R-lens on Gemma-4-31B-it (and Qwen3.6-27B)

A workspace lens maps a mid-layer residual to the final layer with one linear map per layer, then decodes it
with the model's unembedding. The readout evals ask whether a lens reads a concept the model needs but has not yet
said (e.g. *Brazil* when the prompt asks for "the ocean on the coast of the country where Carnival is celebrated").

| Lens | What it is |
|---|---|
| **logit** | the unembedding applied directly to the residual (no map) |
| **J** | the average Jacobian from the layer to the final block (plain backward pass) |
| **R** | the same average with LRP backward rules (the R-lens) |
| **J++** | 8 expert Jacobians per layer (k-means clusters of the residual) combined with signed weights fitted on labelled eval items |

**Metric.** recall@10: an intermediate counts as read at a layer if one of its single-token spellings ranks in the
lens's top 10 (tokens with no letter or digit removed, except eval answers such as `-` or `*`). Items average their
intermediates; the macro averages the four evals with at least 20 items on both models (association, multihop,
multilingual, typo; poetry has 8 Gemma-it items). "Headline" = read at *any* of the 7 readout layers.
All items are the held-out split (`held-out:0`) under each model's correctness filter.

**Which lenses.** Gemma-it: our 64-prompt J++ and R, and a 16-prompt J-lens (fitted for this notebook on the same
16 prompts as the 16-prompt R snapshot, so compare J with *R (16 prompts)*). Qwen: the released lenses
(`koayon/jpp-lenses` J++, `camilablank/workspace-lenses` J and R).

The records are the files in `results/exp12_gemma_lenses/` and its `manifest.json`; numbers here are recomputed from
their `ranks.csv` (bootstrap CIs use this notebook's own seed, so they can differ from the reports in the third decimal).

**At a glance** (4-eval macro, held-out items):
- Gemma-it is weaker than Qwen for every fitted lens: J++ 0.48 vs 0.71 headline. The paper's J++ on *base* Gemma is
  0.58, so the gap is not Gemma as such; base vs instruct and the instruct model's item set are not separated here.
- Per layer (section 2), J++ is the only lens that reads intermediates in Gemma-it's middle layers (0.13-0.24 at
  depths 1/8-5/8, vs at most 0.12 for R and about 0 for J and logit). At the last readout layer every lens reaches
  about 0.4, which is why the any-layer headline barely separates them.
- Gemma's 16-prompt J-lens reads almost nothing before the last readout layer (at most 0.05, below the logit lens at
  depth 6/8); its headline (0.43) matches the logit lens (0.44), as the paper reports for base Gemma (J 34.9 vs
  logit 34.8).
- More or longer fit text does not help J++ (section 4); long records make the R-lens worse by about 4 points.

In [1]:
import json
import re
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
RESULTS = ROOT / "results" / "exp12_gemma_lenses"
EVALS = ["association", "multihop", "multilingual", "typo"]

MODELS = {
    "Gemma-4-31B-it": {"dir": "readouts_gemma", "layers": [8, 15, 22, 30, 38, 45, 52],
                       "lenses": {"jpp_64p": "J++", "jlens_16p": "J (16 prompts)", "rlens_16p": "R (16 prompts)",
                                  "rlens_64p": "R", "logit": "logit"}},
    "Qwen3.6-27B": {"dir": "readouts_qwen", "layers": [8, 16, 24, 32, 40, 48, 56],
                    "lenses": {"jpp_released": "J++", "jlens_camila": "J", "rlens_camila": "R", "logit": "logit"}},
}
COLOURS = {"J++": "#7b3294", "J": "#2c7bb6", "J (16 prompts)": "#2c7bb6", "R": "#1a9641",
           "R (16 prompts)": "#a6d96a", "logit": "#969696"}
MAIN_LENSES = ["J++", "J", "J (16 prompts)", "R (16 prompts)", "R", "logit"]


def load_readouts(model: str) -> dict[str, pd.DataFrame]:
    spec = MODELS[model]
    out = {name: pd.read_csv(RESULTS / spec["dir"] / f"{name}.csv") for name in ("ranks", "top_tokens", "items")}
    for name in ("ranks", "top_tokens"):
        out[name]["lens"] = out[name]["lens"].map(spec["lenses"])
    out["top_tokens"]["top_tokens"] = out["top_tokens"]["top_tokens"].map(json.loads)
    out["items"]["model_top_tokens"] = out["items"]["model_top_tokens"].map(json.loads)
    assert out["items"]["item"].is_unique
    return out


DATA = {model: load_readouts(model) for model in MODELS}
{model: f"{d['items'].shape[0]} items, lenses {sorted(d['ranks']['lens'].unique())}" for model, d in DATA.items()}

{'Gemma-4-31B-it': "244 items, lenses ['J (16 prompts)', 'J++', 'R', 'R (16 prompts)', 'logit']",
 'Qwen3.6-27B': "329 items, lenses ['J', 'J++', 'R', 'logit']"}

## 1. Scorecard
Headline recall@10 per eval and the macro. The paper's numbers are for reference only: its Gemma row is the
**base** model (we use the instruct model), and its Qwen macro includes poetry.

In [2]:
def item_hits(ranks: pd.DataFrame, k: int = 10, by_layer: bool = False) -> pd.Series:
    """Per (lens, eval, item[, layer]): the fraction of the item's intermediates ranked <= k
    (at that layer, or at their best layer)."""
    keys = ["lens", "eval", "item", "intermediate"] + (["layer"] if by_layer else [])
    best = ranks.groupby(keys)["rank"].min()
    group = ["lens", "eval", "item"] + (["layer"] if by_layer else [])
    return (best <= k).groupby(level=group).mean()


def recall_table(ranks: pd.DataFrame, by_layer: bool = False) -> pd.DataFrame:
    """Recall@10 per eval plus the macro over EVALS, as rows of (lens[, layer])."""
    hits = item_hits(ranks, by_layer=by_layer)
    group = ["lens", "eval"] + (["layer"] if by_layer else [])
    per_eval = hits.groupby(level=group).mean().unstack("eval")[EVALS]
    return per_eval.assign(macro=per_eval.mean(axis=1))


PAPER = {"Gemma-4-31B-it": {"J++": 0.58, "J": 0.349, "logit": 0.348},  # base model, poetry dropped
         "Qwen3.6-27B": {"J++": 0.552, "J": 0.357, "R": 0.377, "logit": 0.311}}  # with poetry

rows = []
for model, d in DATA.items():
    table = recall_table(d["ranks"])
    for lens, row in table.iterrows():
        rows.append({"model": model, "lens": lens, **row.round(3).to_dict(),
                     "paper macro": PAPER[model].get(lens, np.nan)})
scorecard = pd.DataFrame(rows).set_index(["model", "lens"])
order = [(m, lens) for m in MODELS for lens in MAIN_LENSES if (m, lens) in scorecard.index]
display(scorecard.loc[order].style.format("{:.3f}", na_rep="").background_gradient(
    subset=EVALS + ["macro"], cmap="Purples", vmin=0, vmax=1))

In [3]:
fig = make_subplots(rows=1, cols=2, subplot_titles=list(MODELS), shared_yaxes=True)
for col, (model, d) in enumerate(DATA.items(), start=1):
    table = recall_table(d["ranks"])
    for lens in [lens for lens in MAIN_LENSES if lens in table.index]:
        fig.add_bar(x=EVALS + ["macro"], y=table.loc[lens, EVALS + ["macro"]], name=lens,
                    marker_color=COLOURS[lens], legendgroup=lens, showlegend=col == 1, row=1, col=col,
                    hovertemplate=f"{lens}<br>%{{x}}: %{{y:.3f}}<extra></extra>")
fig.update_layout(barmode="group", height=420, title="Headline recall@10 (read at any readout layer)",
                  yaxis_title="recall@10", yaxis_range=[0, 1], legend_title="lens")
fig.show()

## 2. Across layers
Recall@10 at each readout layer. Both models are read at depths 1/8 ... 7/8, so the x-axis is depth; hover shows
the layer. Use the dropdown to switch between the macro and single evals.

In [4]:
def per_layer_figure(metric: str = "recall") -> go.Figure:
    views = ["macro"] + EVALS
    fig = make_subplots(rows=1, cols=2, subplot_titles=list(MODELS), shared_yaxes=True)
    traces_per_view = []
    for view in views:
        count = 0
        for col, (model, d) in enumerate(DATA.items(), start=1):
            layers = MODELS[model]["layers"]
            if metric == "recall":
                table = recall_table(d["ranks"], by_layer=True)[view].unstack("layer")
            else:
                ranks = d["ranks"] if view == "macro" else d["ranks"][d["ranks"]["eval"] == view]
                log_rank = np.log10(ranks.groupby(["lens", "eval", "item", "intermediate", "layer"])["rank"].min())
                per_eval = log_rank.groupby(level=["lens", "eval", "layer"]).mean().unstack("eval")
                table = per_eval.mean(axis=1).unstack("layer")
            for lens in [lens for lens in MAIN_LENSES if lens in table.index]:
                fig.add_scatter(
                    x=[f"{i}/8" for i in range(1, 8)], y=table.loc[lens, layers], name=lens, mode="lines+markers",
                    line=dict(color=COLOURS[lens], width=3, dash="dot" if "16 prompts" in lens else "solid"),
                    customdata=layers, legendgroup=lens, showlegend=col == 1, visible=view == "macro",
                    hovertemplate=f"{lens}<br>layer %{{customdata}}: %{{y:.3f}}<extra></extra>", row=1, col=col)
                count += 1
        traces_per_view.append(count)
    buttons = []
    for i, view in enumerate(views):
        visible = sum(([j == i] * n for j, n in enumerate(traces_per_view)), [])
        buttons.append(dict(label=view, method="update", args=[{"visible": visible}]))
    title = "Recall@10 per layer" if metric == "recall" else "Mean log10 rank of the intermediate per layer (lower is better)"
    fig.update_layout(height=430, title=title, updatemenus=[dict(buttons=buttons, x=1.0, y=1.18, xanchor="right")],
                      xaxis_title="depth", xaxis2_title="depth",
                      yaxis_title="recall@10" if metric == "recall" else "log10 rank")
    if metric == "recall":
        fig.update_yaxes(range=[0, 1])
    return fig


per_layer_figure("recall").show()
per_layer_figure("log_rank").show()

## 3. Item browser
Pick a model, eval and item. The heatmap shows each lens (rows) at each layer (columns): colour = log10 rank of the
best intermediate (darker is better), cell text = the lens's top token (✓ when an intermediate is in its top 10).
Hover for the top 5. The model's own top tokens at the readout position are listed above the heatmap.

In [5]:
def item_figure(model: str, item: str) -> go.Figure:
    d, layers = DATA[model], MODELS[model]["layers"]
    lenses = [lens for lens in MAIN_LENSES if lens in set(d["ranks"]["lens"])]
    ranks = d["ranks"][d["ranks"]["item"] == item]
    best = ranks.groupby(["lens", "layer"])["rank"].min().unstack("layer").loc[lenses, layers]
    best_intermediate = ranks.loc[ranks.groupby(["lens", "layer"])["rank"].idxmin()].set_index(["lens", "layer"])["intermediate"]
    tops = d["top_tokens"][d["top_tokens"]["item"] == item].set_index(["lens", "layer"])["top_tokens"]
    text = [[f"{tops[lens, layer][0]!r}{' ✓' if best.loc[lens, layer] <= 10 else ''}" for layer in layers] for lens in lenses]
    hover = [[f"<b>{lens}, layer {layer}</b><br>top 5: {', '.join(repr(t) for t in tops[lens, layer])}"
              f"<br>best intermediate {best_intermediate[lens, layer]!r}: rank {best.loc[lens, layer]:,}"
              for layer in layers] for lens in lenses]
    fig = go.Figure(go.Heatmap(
        z=np.log10(best.to_numpy()), x=[f"L{layer}" for layer in layers], y=lenses, text=text, texttemplate="%{text}",
        hovertext=hover, hoverinfo="text", colorscale="Purples_r", zmin=0, zmax=5,
        colorbar=dict(title="log10 rank", tickvals=[0, 1, 2, 3, 4, 5], ticktext=["1", "10", "100", "1k", "10k", "100k"])))
    fig.update_layout(height=120 + 55 * len(lenses), margin=dict(t=30, b=30), yaxis_autorange="reversed",
                      title=f"{model}: {item}")
    return fig


def show_item(model: str, item: str) -> None:
    row = DATA[model]["items"].set_index("item").loc[item]
    display(HTML(f"<b>{row['eval']}</b> &nbsp; intermediates: <code>{', '.join(json.loads(row['intermediates']))}</code>"
                 f"<br><pre style='white-space:pre-wrap'>{row['prompt'][-600:]}</pre>"
                 f"model's own top 5 at the readout position: <code>{', '.join(map(repr, row['model_top_tokens']))}</code>"))
    item_figure(model, item).show()


model_box = widgets.Dropdown(options=list(MODELS), description="model")
eval_box = widgets.Dropdown(options=EVALS + ["poetry"], description="eval")
item_box = widgets.Dropdown(description="item", layout=widgets.Layout(width="420px"))
output = widgets.Output()


def refresh_items(*_):
    items = DATA[model_box.value]["items"]
    item_box.options = sorted(items.loc[items["eval"] == eval_box.value, "item"])


def redraw(*_):
    output.clear_output(wait=True)
    if item_box.value:
        with output:
            show_item(model_box.value, item_box.value)


model_box.observe(refresh_items, "value")
eval_box.observe(refresh_items, "value")
item_box.observe(redraw, "value")
refresh_items()
display(widgets.HBox([model_box, eval_box, item_box]), output)

Output()

A fixed example, visible without a running kernel (an item held out on both models):

In [6]:
shared_items = set(DATA["Gemma-4-31B-it"]["items"]["item"]) & set(DATA["Qwen3.6-27B"]["items"]["item"])
EXAMPLE = "carnival-ocean" if "carnival-ocean" in shared_items else sorted(shared_items)[0]
for model in MODELS:
    show_item(model, EXAMPLE)

### Where J++ reads what R does not
Per item on Gemma-it: the best rank (any layer) under R vs under J++. Points below the diagonal are items J++ reads
better. Hover for the item; the dotted lines mark rank 10.

In [7]:
def best_rank_scatter(model: str, x_lens: str, y_lens: str) -> go.Figure:
    ranks = DATA[model]["ranks"]
    best = ranks.groupby(["lens", "eval", "item"])["rank"].min()
    table = best.unstack("lens")[[x_lens, y_lens]].reset_index()
    fig = px.scatter(table, x=x_lens, y=y_lens, color="eval", hover_name="item", log_x=True, log_y=True, opacity=0.75,
                     title=f"{model}: best rank over layers, {y_lens} vs {x_lens}")
    fig.add_shape(type="line", x0=1, y0=1, x1=3e5, y1=3e5, line=dict(color="#bbbbbb"))
    fig.add_vline(x=10, line_dash="dot", line_color="#bbbbbb")
    fig.add_hline(y=10, line_dash="dot", line_color="#bbbbbb")
    fig.update_layout(height=480, width=620)
    return fig


best_rank_scatter("Gemma-4-31B-it", "R", "J++").show()

## 4. Fit-data experiments (Gemma-it)
Headline recall@10 of every fitted variant minus the 64-prompt lens of the same kind, with 95% paired item-bootstrap
CIs (items resampled within eval). "B" rows are second fits of the same recipe on different WikiText (fit-to-fit
noise). Short = 128-token prompts; long = 512-token records of at least 3,000 characters.

In [8]:
VARIANTS = [  # (kind, label, ranks file, lens)
    ("J++", "8 short prompts", "eval_v1", "jpp_08p"), ("J++", "16 short prompts", "eval_v1", "jpp_16p"),
    ("J++", "32 short prompts", "eval_v1", "jpp_32p"), ("J++", "64 short prompts, B", "eval_rep_short_v1", "jpp_64p_b"),
    ("J++", "7 long records", "eval_long_v1", "jpp_long_k8_07p"), ("J++", "14 long records", "eval_long_v1", "jpp_long_k8_14p"),
    ("J++", "14 long records, B", "eval_rep_long_v1", "jpp_long_b_k8_14p"),
    ("J++", "28 long records", "eval_rep_long_v1", "jpp_long_k8_28p"),
    ("J++", "14 long records, K=16", "eval_long_v1", "jpp_long_k16_14p"),
    ("J++", "released base-model lens", "eval_transfer_v1", "jpp_base"),
    ("R", "8 short prompts", "eval_v1", "rlens_08p"), ("R", "16 short prompts", "eval_v1", "rlens_16p"),
    ("R", "32 short prompts", "eval_v1", "rlens_32p"), ("R", "64 short prompts, B", "eval_rep_short_v1", "rlens_64p_b"),
    ("R", "7 long records", "eval_long_v1", "rlens_long_07p"), ("R", "14 long records", "eval_long_v1", "rlens_long_14p"),
    ("R", "14 long records, B", "eval_rep_long_v1", "rlens_long_b_14p"),
    ("R", "28 long records", "eval_rep_long_v1", "rlens_long_28p"),
]
REFERENCE = {"J++": "jpp_64p", "R": "rlens_64p"}


def paired_macro_difference(ranks: pd.DataFrame, lens: str, reference: str, resamples: int = 2000,
                            seed: int = 0) -> tuple[float, float, float]:
    """Macro headline recall@10 of lens minus reference, and its 95% CI from resampling items within eval."""
    hits = item_hits(ranks[ranks["lens"].isin([lens, reference])]).unstack("lens")
    diff = {e: (hits.loc[e, lens] - hits.loc[e, reference]).to_numpy() for e in EVALS}
    rng = np.random.default_rng(seed)
    draws = [np.mean([rng.choice(v, len(v)).mean() for v in diff.values()]) for _ in range(resamples)]
    return float(np.mean([v.mean() for v in diff.values()])), *np.percentile(draws, [2.5, 97.5])


rows = []
ranks_by_source = {source: pd.read_csv(RESULTS / source / "ranks.csv") for source in {v[2] for v in VARIANTS}}
for kind, label, source, lens in VARIANTS:
    d, lo, hi = paired_macro_difference(ranks_by_source[source], lens, REFERENCE[kind])
    rows.append({"kind": kind, "variant": label, "lens": lens, "diff": d, "lo": lo, "hi": hi})
variants = pd.DataFrame(rows)

fig = make_subplots(rows=1, cols=2, subplot_titles=["J++ minus jpp_64p", "R minus rlens_64p"], shared_xaxes=True,
                    horizontal_spacing=0.25)
for col, kind in enumerate(["J++", "R"], start=1):
    part = variants[variants["kind"] == kind].iloc[::-1]
    fig.add_scatter(x=part["diff"], y=part["variant"], mode="markers", marker=dict(size=10, color=COLOURS[kind]),
                    error_x=dict(type="data", symmetric=False, array=part["hi"] - part["diff"],
                                 arrayminus=part["diff"] - part["lo"], color=COLOURS[kind]),
                    customdata=part[["lo", "hi", "lens"]], showlegend=False, row=1, col=col,
                    hovertemplate="%{customdata[2]}: %{x:+.3f} [%{customdata[0]:+.3f}, %{customdata[1]:+.3f}]<extra></extra>")
    fig.add_vline(x=0, line_color="#888888", row=1, col=col)
fig.update_layout(height=470, title="Headline recall@10 relative to the 64-prompt lens (95% CI)")
fig.update_xaxes(title="difference in recall@10", zeroline=False)
fig.show()

## 5. Brew gate: does Gemma-it make no-CoT brew errors?
Accuracy of the no-CoT colour answer by number of hops, from `brew_gate/gate.txt` (Qwen3.6-27B from exp11 for
reference). The pre-registered gate looked at 2 hops on the three-column format.

In [9]:
gate_text = (RESULTS / "brew_gate" / "gate.txt").read_text()
accuracy = []
for bank, block in re.findall(r"## bank (\w+)\n(.*?)(?=\n## |\ngate)", gate_text, flags=re.S):
    for h, acc in re.findall(r"h=(\d) n=\d+ accuracy ([\d.]+)", block):
        accuracy.append({"format": {"test": "three columns", "single_test": "single table"}[bank],
                         "model": "Gemma-4-31B-it", "hops": int(h), "accuracy": float(acc)})
qwen_reference = {"three columns": [1.00, 0.12, 0.05], "single table": [1.00, 0.76, 0.35]}
for fmt, values in qwen_reference.items():
    accuracy += [{"format": fmt, "model": "Qwen3.6-27B (exp11)", "hops": h, "accuracy": a} for h, a in zip([1, 2, 3], values)]
fig = px.line(pd.DataFrame(accuracy), x="hops", y="accuracy", color="model", facet_col="format", markers=True,
              color_discrete_map={"Gemma-4-31B-it": "#7b3294", "Qwen3.6-27B (exp11)": "#e66101"})
fig.update_layout(height=360, title="No-CoT brew accuracy by hops")
fig.update_xaxes(dtick=1)
fig.update_yaxes(range=[0, 1.05])
fig.show()

families = pd.DataFrame(
    [{"hops": int(h), "family": f, "share of errors": float(o), "lo": float(lo), "hi": float(hi), "chance": float(n)}
     for h, block in re.findall(r"h=(\d) n=\d+ accuracy [\d.]+.*?\n((?:    \w+ .*\n)+)", gate_text.split("## bank single_test")[0])
     for f, o, lo, hi, n in re.findall(r"(\w+)\s+([\d.]+) \[([\d.]+), ([\d.]+)\]\s+null ([\d.]+)", block)])
fig = px.bar(families, x="family", y="share of errors", facet_col="hops", color_discrete_sequence=["#7b3294"],
             error_y=families["hi"] - families["share of errors"], error_y_minus=families["share of errors"] - families["lo"],
             title="Gemma-it three-column errors: share matching each shortcut, vs chance (black ticks)")
for i, h in enumerate(sorted(families["hops"].unique()), start=1):
    part = families[families["hops"] == h]
    fig.add_scatter(x=part["family"], y=part["chance"], mode="markers", marker=dict(symbol="line-ew-open", size=28, color="black"),
                    showlegend=False, row=1, col=i, hovertemplate="chance %{y:.3f}<extra></extra>")
fig.update_layout(height=380)
fig.show()